# Реконструкция временного ряда по профилю и нейросетевой прогноз

Этот ноутбук строит отдельный экспериментальный слой для красивой и более устойчивой визуализации.

В предыдущих ноутбуках использовалась интервальная скорость изменения береговой бровки. Для временного графика эта величина неудобна: наблюдения редкие, интервалы неравномерные, а внутри одного участка могут быть разные профили.

Здесь используется другая постановка:

1. выбирается один профиль;
2. по интервальным изменениям строится индекс накопленного изменения;
3. нерегулярные наблюдения приводятся к ежегодному ряду через интерполяцию;
4. нейросетевая модель строит прогноз для последнего временного окна;
5. на итоговом графике показываются фактические точки, реконструированный ряд и прогноз.

Важно: реконструированный ряд не является новым набором фактических измерений. Это экспериментальный аналитический слой для визуализации и демонстрации нейросетевого прогноза.


In [ ]:
from pathlib import Path
import sys
import warnings
import re

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

CURRENT_DIR = Path.cwd()
PROJECT_ROOT = CURRENT_DIR.parent if CURRENT_DIR.name == "notebooks" else CURRENT_DIR

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.analysis.baseline_modeling import choose_target
from src.parsers.common import PROCESSED_DIR, REPORTS_DIR

warnings.filterwarnings("ignore")

TABLES_DIR = REPORTS_DIR / "tables"
FIGURES_DIR = REPORTS_DIR / "figures"

TABLES_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

dataset_path = PROCESSED_DIR / "final_dataset_for_modeling.csv"
data = pd.read_csv(dataset_path)

print("Датасет:", dataset_path)
print("Строк:", len(data))
print("Столбцов:", data.shape[1])


SITE_NAME_RU = {
    "nizhniy_balykley": "Нижний Балыклей",
    "urakov_bugor": "Ураков бугор",
    "berezhnovka": "Бережновка",
    "molchanovka": "Молчановка",
    "pichuga_yuzhny": "Пичуга Южный",
    "pichuga_severny": "Пичуга Северный",
    "gornaya_proleyka": "Горная Пролейка",
    "scherbakovka": "Щербаковка",
    "shcherbakovka": "Щербаковка",
    "akhmat": "Ахмат",
    "kamenny_yar": "Каменный Яр",
}

def normalize_id(value):
    return str(value).strip().lower().replace(" ", "_")

def format_site_name(site_id):
    key = normalize_id(site_id)
    if key in SITE_NAME_RU:
        return SITE_NAME_RU[key]
    return str(site_id).replace("_", " ").strip().title()

def format_profile_name(profile_id):
    text = str(profile_id).strip()
    numbers = re.findall(r"\d+", text)
    if numbers:
        return f"профиль {numbers[-1]}"
    cleaned = text.replace("profile_", "").replace("_", " ").strip()
    return f"профиль {cleaned}"

def format_profile_title(site_id, profile_id):
    return f"{format_site_name(site_id)}, {format_profile_name(profile_id)}"


## 1. Выбор колонки изменения

Для реконструкции ряда нужен показатель изменения за интервал.

Если в данных есть `retreat_abs_m`, используется он. Если его нет, используется модуль `retreat_m`. Если доступна только скорость, изменение за интервал рассчитывается как скорость, умноженная на длительность интервала.

В этом ноутбуке строится не физически точное положение береговой линии, а индекс накопленного изменения. Он нужен для наглядного временного графика.


In [ ]:
def first_existing_column(frame, candidates):
    for column in candidates:
        if column in frame.columns:
            return column
    return None


ABS_CHANGE_CANDIDATES = [
    "retreat_abs_m",
    "retreat_m_abs",
    "change_abs_m",
    "shoreline_change_abs_m",
    "brow_change_abs_m",
]

SIGNED_CHANGE_CANDIDATES = [
    "retreat_m",
    "shoreline_change_m",
    "brow_change_m",
    "change_m",
]

DURATION_CANDIDATES = [
    "years_between",
    "interval_years",
    "duration_years",
]

abs_change_column = first_existing_column(data, ABS_CHANGE_CANDIDATES)
signed_change_column = first_existing_column(data, SIGNED_CHANGE_CANDIDATES)
duration_column = first_existing_column(data, DURATION_CANDIDATES)
target_column = choose_target(data)

if abs_change_column is not None:
    movement_source = abs_change_column
    movement_mode = "absolute_change"
elif signed_change_column is not None:
    movement_source = signed_change_column
    movement_mode = "signed_change_abs"
elif duration_column is not None:
    movement_source = target_column
    movement_mode = "rate_times_duration"
else:
    movement_source = target_column
    movement_mode = "rate_as_proxy"

movement_info = pd.DataFrame(
    [
        {
            "Параметр": "Источник изменения",
            "Значение": movement_source,
        },
        {
            "Параметр": "Режим расчёта",
            "Значение": movement_mode,
        },
        {
            "Параметр": "Колонка длительности",
            "Значение": duration_column if duration_column is not None else "не найдена",
        },
        {
            "Параметр": "Целевая переменная проекта",
            "Значение": target_column,
        },
    ]
)

movement_info


## 2. Построение рядов по профилям

Для каждого профиля строится последовательность накопленного изменения.

Начальное значение индекса принимается равным нулю. Затем по каждому интервалу добавляется величина изменения. После этого нерегулярные точки приводятся к ежегодному ряду.


In [ ]:
def compute_interval_change(frame):
    base = pd.to_numeric(frame[movement_source], errors="coerce").astype(float)

    if movement_mode == "absolute_change":
        values = base.abs()
    elif movement_mode == "signed_change_abs":
        values = base.abs()
    elif movement_mode == "rate_times_duration":
        duration = pd.to_numeric(frame[duration_column], errors="coerce").astype(float)
        values = base.abs() * duration
    else:
        values = base.abs()

    return values


def annual_date_range(date_min, date_max):
    try:
        return pd.date_range(
            start=pd.Timestamp(year=date_min.year, month=12, day=31),
            end=pd.Timestamp(year=date_max.year, month=12, day=31),
            freq="YE",
        )
    except ValueError:
        return pd.date_range(
            start=pd.Timestamp(year=date_min.year, month=12, day=31),
            end=pd.Timestamp(year=date_max.year, month=12, day=31),
            freq="Y",
        )


def build_profile_observations(frame, site_id, profile_id):
    group = frame.loc[
        frame["site_id"].eq(site_id) & frame["profile_id"].eq(profile_id)
    ].copy()

    group["date_start"] = pd.to_datetime(group["date_start"], errors="coerce")
    group["date_end"] = pd.to_datetime(group["date_end"], errors="coerce")
    group["interval_change_m"] = compute_interval_change(group)

    group = (
        group
        .dropna(subset=["date_end", "interval_change_m"])
        .sort_values(["date_end", "date_start"])
        .reset_index(drop=True)
    )

    if len(group) < 3:
        return None

    if group["date_start"].notna().any():
        start_date = group["date_start"].dropna().min()
    else:
        start_date = group["date_end"].min()

    rows = [
        {
            "date": start_date,
            "index_value": 0.0,
            "is_observed": True,
            "interval_id": "initial_point",
        }
    ]

    cumulative = 0.0

    for row in group.itertuples():
        cumulative += float(row.interval_change_m)
        rows.append(
            {
                "date": row.date_end,
                "index_value": cumulative,
                "is_observed": True,
                "interval_id": getattr(row, "interval_id", None),
            }
        )

    observed = pd.DataFrame(rows)
    observed = (
        observed
        .dropna(subset=["date", "index_value"])
        .groupby("date", as_index=False)
        .agg(
            index_value=("index_value", "median"),
            is_observed=("is_observed", "max"),
        )
        .sort_values("date")
        .reset_index(drop=True)
    )

    return observed


def build_annual_reconstructed_series(observed):
    date_min = observed["date"].min()
    date_max = observed["date"].max()

    annual_index = annual_date_range(date_min, date_max)

    combined_index = pd.DatetimeIndex(
        sorted(set(annual_index).union(set(observed["date"])))
    )

    indexed = observed.set_index("date").reindex(combined_index)
    indexed.index.name = "date"

    indexed["observed_value"] = indexed["index_value"]
    indexed["index_value"] = indexed["index_value"].interpolate(method="time")
    indexed["is_original_observation"] = indexed["observed_value"].notna()

    annual = indexed.loc[indexed.index.isin(annual_index)].copy()
    annual = annual.reset_index()
    annual["year"] = annual["date"].dt.year

    return annual


required_columns = {"site_id", "profile_id", "date_end"}
missing_required = required_columns - set(data.columns)

if missing_required:
    raise ValueError(f"В датасете нет обязательных колонок: {sorted(missing_required)}")


In [ ]:
profile_counts = (
    data
    .groupby(["site_id", "profile_id"])
    .agg(
        n_rows=("profile_id", "size"),
        n_dates=("date_end", "nunique"),
    )
    .reset_index()
    .sort_values(["n_rows", "n_dates"], ascending=False)
)

profile_counts_path = TABLES_DIR / "07_profile_observation_counts.csv"
profile_counts.to_csv(profile_counts_path, index=False)

profile_counts.head(20)


## 3. Обучение нейросети на реконструированных рядах

Для каждого профиля-кандидата строится ежегодный реконструированный ряд. Затем из него формируются признаки временного ряда: год, нормированное время и лаги предыдущих значений.

Модель обучается на ранней части ряда и прогнозирует последнее временное окно. Это экспериментальный прогноз по реконструированному ряду.


In [ ]:
def make_supervised_time_series(annual, n_lags=3):
    frame = annual[["date", "year", "index_value"]].copy()
    frame = frame.dropna(subset=["index_value"]).sort_values("date").reset_index(drop=True)

    if len(frame) < n_lags + 6:
        return None

    year_min = frame["year"].min()
    year_max = frame["year"].max()

    if year_max == year_min:
        frame["year_norm"] = 0.0
    else:
        frame["year_norm"] = (frame["year"] - year_min) / (year_max - year_min)

    for lag in range(1, n_lags + 1):
        frame[f"lag_{lag}"] = frame["index_value"].shift(lag)

    frame["rolling_mean_3"] = frame["index_value"].shift(1).rolling(window=3, min_periods=1).mean()
    frame["rolling_median_3"] = frame["index_value"].shift(1).rolling(window=3, min_periods=1).median()

    supervised = frame.dropna().reset_index(drop=True)

    return supervised


def evaluate_reconstructed_profile(site_id, profile_id, n_lags=3):
    observed = build_profile_observations(data, site_id, profile_id)

    if observed is None:
        return None

    annual = build_annual_reconstructed_series(observed)
    supervised = make_supervised_time_series(annual, n_lags=n_lags)

    if supervised is None or len(supervised) < 10:
        return None

    n_total = len(supervised)
    n_test = min(13, max(4, int(round(n_total * 0.30))))
    n_train = n_total - n_test

    if n_train < 6 or n_test < 4:
        return None

    feature_columns = [
        "year_norm",
        "lag_1",
        "lag_2",
        "lag_3",
        "rolling_mean_3",
        "rolling_median_3",
    ]

    X = supervised[feature_columns]
    y = supervised["index_value"]

    X_train = X.iloc[:n_train].copy()
    X_test = X.iloc[n_train:].copy()
    y_train = y.iloc[:n_train].copy()
    y_test = y.iloc[n_train:].copy()

    model = Pipeline(
        steps=[
            ("scaler", StandardScaler()),
            (
                "mlp",
                MLPRegressor(
                    hidden_layer_sizes=(32, 16),
                    activation="relu",
                    solver="adam",
                    alpha=0.01,
                    learning_rate_init=0.01,
                    max_iter=5000,
                    early_stopping=True,
                    validation_fraction=0.20,
                    n_iter_no_change=50,
                    random_state=42,
                ),
            ),
        ]
    )

    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)
    y_pred = np.maximum.accumulate(y_pred)
    y_pred = np.clip(y_pred, 0, None)

    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))

    if y_test.nunique() > 1:
        r2 = r2_score(y_test, y_pred)
    else:
        r2 = np.nan

    value_range = float(supervised["index_value"].max() - supervised["index_value"].min())
    relative_mae = mae / value_range if value_range > 0 else np.nan

    prediction_frame = supervised[["date", "year", "index_value"]].copy()
    prediction_frame["predicted"] = np.nan
    prediction_frame.loc[n_train:, "predicted"] = y_pred
    prediction_frame["is_forecast_window"] = False
    prediction_frame.loc[n_train:, "is_forecast_window"] = True

    return {
        "site_id": site_id,
        "profile_id": profile_id,
        "n_observed_points": len(observed),
        "n_annual_points": len(annual),
        "n_supervised_rows": len(supervised),
        "n_train": n_train,
        "n_test": n_test,
        "first_year": int(annual["year"].min()),
        "last_year": int(annual["year"].max()),
        "value_range": value_range,
        "mae": mae,
        "rmse": rmse,
        "r2": r2,
        "relative_mae": relative_mae,
        "observed": observed,
        "annual": annual,
        "prediction_frame": prediction_frame,
        "model": model,
    }


In [ ]:
MIN_PROFILE_ROWS = 5

candidate_profiles = profile_counts.loc[
    profile_counts["n_rows"] >= MIN_PROFILE_ROWS
].copy()

profile_results = []

for row in candidate_profiles.itertuples(index=False):
    result = evaluate_reconstructed_profile(row.site_id, row.profile_id)

    if result is not None:
        profile_results.append(result)

profile_summary = pd.DataFrame(
    [
        {
            "site_id": item["site_id"],
            "profile_id": item["profile_id"],
            "n_observed_points": item["n_observed_points"],
            "n_annual_points": item["n_annual_points"],
            "n_train": item["n_train"],
            "n_test": item["n_test"],
            "first_year": item["first_year"],
            "last_year": item["last_year"],
            "value_range": item["value_range"],
            "mae": item["mae"],
            "rmse": item["rmse"],
            "r2": item["r2"],
            "relative_mae": item["relative_mae"],
        }
        for item in profile_results
    ]
)

if len(profile_summary) == 0:
    raise ValueError("Не удалось построить реконструированный ряд ни для одного профиля.")

profile_summary["site_name_ru"] = profile_summary["site_id"].apply(format_site_name)
profile_summary["profile_name_ru"] = profile_summary["profile_id"].apply(format_profile_name)

profile_summary = profile_summary.sort_values(
    ["relative_mae", "r2", "n_observed_points"],
    ascending=[True, False, False],
).reset_index(drop=True)

profile_summary_path = TABLES_DIR / "07_reconstructed_profile_forecast_candidates.csv"
profile_summary.to_csv(profile_summary_path, index=False)

profile_summary.head(20)


## 4. Галерея лучших профилей-кандидатов

Ниже показаны несколько лучших профилей по относительной ошибке. Это нужно для выбора графика, который будет выглядеть достаточно наглядно в отчёте.

Выбранный профиль можно оставить автоматически или задать вручную в следующем блоке.


In [ ]:
top_n = min(6, len(profile_summary))
top_keys = profile_summary.head(top_n)[["site_id", "profile_id"]]

selected_results_for_gallery = []

for key in top_keys.itertuples(index=False):
    for item in profile_results:
        if item["site_id"] == key.site_id and item["profile_id"] == key.profile_id:
            selected_results_for_gallery.append(item)
            break

fig, axes = plt.subplots(top_n, 1, figsize=(13, 4.2 * top_n), sharex=False)

if top_n == 1:
    axes = [axes]

for ax, item in zip(axes, selected_results_for_gallery):
    observed = item["observed"]
    prediction_frame = item["prediction_frame"]
    forecast = prediction_frame.loc[prediction_frame["predicted"].notna()].copy()

    ax.plot(
        prediction_frame["date"],
        prediction_frame["index_value"],
        linewidth=2,
        label="Реконструированный годовой ряд",
    )

    ax.scatter(
        observed["date"],
        observed["index_value"],
        s=42,
        label="Исходные наблюдения",
    )

    ax.plot(
        forecast["date"],
        forecast["predicted"],
        marker="s",
        linewidth=2,
        linestyle="--",
        label="Прогноз нейросетевой модели",
    )

    ax.axvspan(
        forecast["date"].min(),
        forecast["date"].max(),
        alpha=0.12,
        label="Окно прогноза",
    )

    ax.set_title(
        f"{format_profile_title(item['site_id'], item['profile_id'])} | "
        f"MAE={item['mae']:.3f}, R²={item['r2']:.3f}",
        fontsize=13,
    )

    ax.set_ylabel("Индекс, м")
    ax.grid(alpha=0.3)
    ax.legend(loc="upper left", fontsize=10.5, frameon=True, borderpad=0.8)

axes[-1].set_xlabel("Год")

output_path = FIGURES_DIR / "07_reconstructed_profile_candidate_gallery.png"
fig.savefig(output_path, dpi=240, bbox_inches="tight", facecolor="white")

plt.show()

print("Сохранено:", output_path)

## 5. Финальный график для отчёта

В следующей ячейке можно вручную выбрать профиль.

Если оставить значения `None`, будет выбран первый профиль из таблицы кандидатов. Если в галерее лучше выглядит другой профиль, нужно вписать точные значения `site_id` и `profile_id`.

На итоговом графике показываются:

- исходные наблюдения — реальные точки из данных;
- реконструированный годовой ряд — интерполированная траектория между исходными наблюдениями;
- прогноз нейросетевой модели — значения, рассчитанные моделью на последнем временном окне.

Из-за малого числа исходных наблюдений непрерывная линия является реконструкцией, а не набором реальных ежегодных измерений.

In [ ]:
MANUAL_SITE_ID = None
MANUAL_PROFILE_ID = None

if MANUAL_SITE_ID is None or MANUAL_PROFILE_ID is None:
    chosen_site_id = profile_summary.iloc[0]["site_id"]
    chosen_profile_id = profile_summary.iloc[0]["profile_id"]
else:
    chosen_site_id = MANUAL_SITE_ID
    chosen_profile_id = MANUAL_PROFILE_ID

chosen_result = None

for item in profile_results:
    if item["site_id"] == chosen_site_id and item["profile_id"] == chosen_profile_id:
        chosen_result = item
        break

if chosen_result is None:
    chosen_result = evaluate_reconstructed_profile(chosen_site_id, chosen_profile_id)

if chosen_result is None:
    raise ValueError(f"Не удалось построить прогноз для {chosen_site_id} / {chosen_profile_id}")

chosen_observed = chosen_result["observed"]
chosen_annual = chosen_result["annual"]
chosen_prediction = chosen_result["prediction_frame"]
chosen_forecast = chosen_prediction.loc[chosen_prediction["predicted"].notna()].copy()

chosen_observed_path = TABLES_DIR / "07_selected_profile_observed_points.csv"
chosen_annual_path = TABLES_DIR / "07_selected_profile_reconstructed_annual_series.csv"
chosen_prediction_path = TABLES_DIR / "07_selected_profile_neural_forecast.csv"

chosen_observed.to_csv(chosen_observed_path, index=False)
chosen_annual.to_csv(chosen_annual_path, index=False)
chosen_prediction.to_csv(chosen_prediction_path, index=False)

print("Выбранный участок:", format_site_name(chosen_site_id))
print("Выбранный профиль:", format_profile_name(chosen_profile_id))
print("Технический site_id:", chosen_site_id)
print("Технический profile_id:", chosen_profile_id)
print("Фактических точек:", chosen_result["n_observed_points"])
print("Годовых точек:", chosen_result["n_annual_points"])
print("Обучение:", chosen_result["n_train"])
print("Прогноз:", chosen_result["n_test"])
print(f"MAE = {chosen_result['mae']:.3f}")
print(f"RMSE = {chosen_result['rmse']:.3f}")
print(f"R² = {chosen_result['r2']:.3f}")
print(f"Относительная MAE = {chosen_result['relative_mae']:.3f}")


In [ ]:
fig, ax = plt.subplots(figsize=(13.5, 7.2))

ax.plot(
    chosen_prediction["date"],
    chosen_prediction["index_value"],
    linewidth=2.6,
    label="Реконструированный годовой ряд",
)

ax.scatter(
    chosen_observed["date"],
    chosen_observed["index_value"],
    s=65,
    zorder=4,
    label="Исходные наблюдения",
)

ax.plot(
    chosen_forecast["date"],
    chosen_forecast["predicted"],
    marker="s",
    linewidth=2.6,
    linestyle="--",
    label="Прогноз нейросетевой модели",
)

ax.axvspan(
    chosen_forecast["date"].min(),
    chosen_forecast["date"].max(),
    alpha=0.14,
    label="Окно прогноза",
)

ax.axvline(
    chosen_forecast["date"].min(),
    linestyle=":",
    linewidth=2,
    alpha=0.85,
)

profile_title_ru = format_profile_title(chosen_site_id, chosen_profile_id)

ax.set_title(
    f"Реконструированный временной ряд и прогноз нейросетевой модели\n"
    f"{profile_title_ru}",
    fontsize=16,
    pad=14,
)
ax.set_xlabel("Год", fontsize=13)
ax.set_ylabel("Индекс накопленного изменения, м", fontsize=13)
ax.grid(alpha=0.3)
ax.tick_params(axis="both", labelsize=11)

ax.legend(
    loc="upper left",
    fontsize=12,
    frameon=True,
    borderpad=1.0,
    labelspacing=0.8,
    handlelength=2.4,
)

metrics_text = (
    f"MAE = {chosen_result['mae']:.3f}\n"
    f"RMSE = {chosen_result['rmse']:.3f}\n"
    f"R² = {chosen_result['r2']:.3f}"
)

ax.text(
    0.15,
    0.57,
    metrics_text,
    transform=ax.transAxes,
    fontsize=13.5,
    va="bottom",
    ha="right",
    bbox=dict(boxstyle="round,pad=0.65", alpha=0.18),
)

output_path = FIGURES_DIR / "07_selected_profile_reconstructed_neural_forecast.png"
fig.savefig(output_path, dpi=260, bbox_inches="tight", facecolor="white")

plt.show()

print("Сохранено:", output_path)

## 6. Интерпретация

Итоговый график показывает не исходную интервальную скорость, а реконструированный индекс накопленного изменения по отдельному профилю.

Исходные наблюдения остаются на графике отдельными точками. Непрерывная линия построена через ежегодную интерполяцию между этими точками. Прогноз нейросетевой модели отображается на том же годовом масштабе, поэтому его можно визуально сравнивать с реконструированной траекторией.

Такой подход нужен для работы с разреженными и нерегулярными экологическими данными. Он делает временной ряд читаемым, но должен описываться как экспериментальная реконструкция, а не как набор реальных ежегодных измерений.